[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/quizzes/quiz_19_containers_docker.ipynb)

# 🧪 Module 19 Quiz — Containers & Docker

Five recall questions to check what stuck after Module 19.

- **Format:** multiple choice; each question has one correct answer.
- **Time:** ~10 minutes.
- **How to use:** read the question, decide which answer you think is correct, *then* click the **Answer + reasoning** block. Don't peek.

---

### Q1 — Image vs container

What is the relationship between a Docker **image** and a **container**?

- **A.** They are two names for the same thing
- **B.** An image is a running container that has been paused
- **C.** An image is an immutable, layered filesystem template; a container is a running (or stopped) *instance* of it with its own writable layer
- **D.** A container is the file you download; an image is what appears in `docker ps`

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: C.**

The image is the read-only recipe-plus-ingredients: an ordered stack of filesystem layers plus config (env, default command). `docker run` creates a container from it — the same layers, topped with one thin **writable layer**, wrapped in namespaces and cgroups, with a process inside. Delete the container and the image is untouched; run the image twice and you get two independent containers.
</details>

### Q2 — Build cache

Your `Dockerfile` starts with `COPY . .` and then runs `RUN pip install -r requirements.txt`. Every time you edit one line of your source code, the rebuild reinstalls **all** dependencies. Why?

- **A.** pip ignores its own cache inside containers
- **B.** A layer's cache is invalidated when its inputs change — and everything *after* an invalidated layer must rebuild too; `COPY . .` changes on every code edit, dragging the `pip install` layer with it
- **C.** Docker always rebuilds every layer from scratch
- **D.** `requirements.txt` gets a new timestamp on every build

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: B.**

The build cache works layer by layer: if an instruction and its inputs are unchanged, the cached layer is reused — but the *first* changed layer invalidates everything below it. The fix is ordering by change frequency: `COPY requirements.txt .` → `RUN pip install -r requirements.txt` → *then* `COPY . .`. Now a code edit only rebuilds the cheap final layers, and dependencies reinstall only when `requirements.txt` itself changes. This one reordering is the single biggest Dockerfile speed-up in data-science images.
</details>

### Q3 — Containers vs virtual machines

Why does a container start in milliseconds while a virtual machine takes seconds to minutes?

- **A.** Containers are compiled ahead of time
- **B.** Containers skip the filesystem entirely and run from RAM
- **C.** A container is just a normal process on the **host's kernel**, isolated by namespaces and cgroups — there is no guest OS to boot
- **D.** VMs are throttled by license checks

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: C.**

A VM boots a whole operating system with its own kernel on simulated hardware. A container boots nothing: the kernel is already running (the host's), and "starting a container" just means starting a process with its own namespaces (isolated view of processes, filesystem, network) and cgroup limits. That's also the key *consequence*: all containers share the host kernel — which is why Docker on macOS/Windows quietly runs a Linux VM under the hood.
</details>

### Q4 — Persisting data

Your Jupyter container writes trained models to `/workspace/models`. After `docker rm`, they're gone. What's the standard fix?

- **A.** Commit the container to a new image after every training run
- **B.** Mount storage into the container — a bind mount of a host directory, or a named volume — so writes land outside the container's ephemeral writable layer
- **C.** Increase the size of the writable layer
- **D.** Run the container with `--restart always` so it's never removed

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: B.**

The container's writable layer dies with the container — that's by design (containers are disposable). Anything that must outlive it goes through a mount: a **bind mount** (`-v $(pwd)/models:/workspace/models`) when you want the files visible in your host project, a **named volume** when Docker should manage the storage (databases, HF model caches). Committing containers to images (A) turns data into image layers — bloated, unshareable, and wrong.
</details>

### Q5 — Exit code 137

A training container keeps dying with exit code **137**, and `docker inspect` shows `"OOMKilled": true`. What happened?

- **A.** The Python process raised `MemoryError` and exited
- **B.** The kernel's OOM killer terminated the process (SIGKILL, 128 + 9 = 137) because the container hit its **cgroup memory limit**
- **C.** A dependency was missing, so the entrypoint crashed
- **D.** Docker restarted the container to apply a new image version

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: B.**

Memory limits are enforced by cgroups, and the enforcement is a kernel kill, not a Python exception — your process never gets a chance to raise `MemoryError`. Exit code 137 = 128 + signal 9 (SIGKILL). Fixes: raise the limit (`--memory`), shrink the workload (batch size, chunked loading), or add swap headroom. The same signature (137) appears in Kubernetes and CI runners — it's worth recognising on sight.
</details>

---

## How did you do?

- **5/5** — you've got it. Move on.
- **3-4/5** — solid. Re-read any chapter you were unsure about, then move on.
- **0-2/5** — re-do the module's lab notebook before continuing.

🚀 **Next:** you've reached the end of the course — put containers to work. Containerize one of your own course projects with the [`dockerfiles-for-data-science.md`](../19_containers_docker/dockerfiles-for-data-science.md) playbook, then take it through [Module 14's tutorial](../14_cicd/tutorial.md) to ship it for real.